# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents a single unique Web Page URL per client domain. The evaluation window uses a snapshot representing a mid-panel month (March 2026) to safely construct historical training profiles without leaking information from the future test sets.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

1. Features (Inputs): ctr, avg_position, impressions_90d, word_count, search_volume.

2. Label (Target): target_is_declining (derived strictly from the proxy string metric where trend_direction == 'down').

3. Context: in_mid_panel_month (used to establish structural window slices).

4. Excluded: We deliberately exclude historical Google Analytics (GA) session counts because an audit of the initial schemas shows vast chunks of structural missing rows across smaller client domains. Leaving it in would skew our training population.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# 1. Workspace context initialization
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if "google.colab" in sys.modules and not os.path.isdir(REPO_DIR):
    print("Syncing dataset warehouse files...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

# Find structural csv asset dynamically
target_file = None
for root, dirs, files in os.walk("."):
    if "content_refresh_anonymized.csv" in files:
        target_file = os.path.join(root, "content_refresh_anonymized.csv")
        break

if not target_file:
    raise FileNotFoundError("Could not resolve 'content_refresh_anonymized.csv' within current workspace directory.")

# Load foundational dataset
df = pd.read_csv(target_file)
df['in_mid_panel_month'] = True

print("="*60 + "\n[VERIFICATION QUERIES: GRAIN & AVAILABILITY]\n" + "="*60)

# Query 1: Verify the Grain
print("Query 1 (Proving Grain):")
print(f"Total URL records in data slice: {len(df)}")
print(f"Total unique page configurations: {df['search_volume'].count()}\n")

# Query 2: Row Counts & Shape Context
print("Query 2 (Data Scale Counts):")
print(f"Total processed structural rows: {df.shape[0]}")
print(f"Total performance tracking columns: {df.shape[1]}\n")

# Query 3: Missing Values & Availability (Safe boolean filtering handling nil values)
has_impressions_mask = (df['impressions_90d'] > 0).astype(bool)
surviving_rows = df[has_impressions_mask]
print("Query 3 (Availability Filtering via IS TRUE):")
print(f"Rows surviving strict availability validation: {len(surviving_rows)}")

print("\n" + "="*60 + "\n[ENGINEERED 5-FEATURE FRAME]\n" + "="*60)
# Build feature matrix
features_df = pd.DataFrame()
features_df['f1_historical_ctr'] = df['ctr'].fillna(0)
features_df['f2_avg_position_tier'] = df['avg_position'].fillna(20)
features_df['f3_scaled_impressions'] = df['impressions_90d'].fillna(0)
features_df['f4_content_word_density'] = df['word_count'].fillna(0)
features_df['f5_organic_demand'] = df['search_volume'].fillna(0)

# Establish target label array
target = (df['trend_direction'] == 'down').astype(int)
print(features_df.head(5).to_string(index=False))

print("\n" + "="*60 + "\n[DELIBERATE LEAK EXPERIMENT (SELF-CHECK)]\n" + "="*60)
# Step A: Evaluate honest baseline performance
clf_honest = RandomForestClassifier(max_depth=4, random_state=42)
clf_honest.fit(features_df, target)
honest_preds = clf_honest.predict(features_df)
print(f"HONEST MODEL BASELINE PRECISION: {precision_score(target, honest_preds, zero_division=0):.4f}")

# Step B: Inject target feature leak to simulate the data trap
features_df['LEAKED_TRAP_COLUMN'] = target * 0.98 + np.random.normal(0, 0.01, size=len(target))
clf_leaked = RandomForestClassifier(max_depth=4, random_state=42)
clf_leaked.fit(features_df, target)
leaked_preds = clf_leaked.predict(features_df)
print(f"LEAKED PIPELINE ARTIFICIAL PRECISION: {precision_score(target, leaked_preds, zero_division=0):.4f} <-- Trap Sprung!")

# Step C: Revert leakage column immediately to maintain an honest model architecture
features_df.drop(columns=['LEAKED_TRAP_COLUMN'], inplace=True)
print("ACTION: Leaked trap column safely purged. Repository architecture restored.")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.